# 03 — Compatibilização areal para a malha de 2022

**Objetivo:** interpolar as contagens de 2000 e 2010 para os limites exatos dos setores censitários de 2022,
usando ponderação por área de interseção (areal weighting), com normalização que preserva os totais municipais.
Núcleo do algoritmo de compatibilização.

**Entradas:**
- `output/02_padronizado/<slug_cidade>_<ano>.gpkg` (15 arquivos)
- `output/01_diagnostico/totais_baseline_por_cidade_ano.csv` (para validação)

**Saídas:**
- `output/03_compatibilizado/<slug_cidade>_contagens.gpkg` (5 arquivos — geometria 2022 + colunas `V*_2000`,
  `V*_2010`, `V*_2022`)

**Depende de:** `02_padronizacao_e_correcao_geometrica.ipynb`

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "utils.py").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

import geopandas as gpd
import pandas as pd

from utils import CIDADES, OUTPUT_DIR, CRS_METRICO, CRS_WGS84, VARS_EXTENSIVAS, VAR_RENDA, PESO_RENDA

IN_DIR = OUTPUT_DIR / "02_padronizado"
OUT_DIR = OUTPUT_DIR / "03_compatibilizado"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LIMIAR_SLIVER_M2 = 5.0  # interseções menores que isso são ruído de topologia, não sobreposição real

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

baseline = pd.read_csv(OUTPUT_DIR / "01_diagnostico" / "totais_baseline_por_cidade_ano.csv")
baseline


## 2. Função de compatibilização areal

Para uma cidade e um ano de origem (2000 ou 2010):

1. Reprojeta origem e destino (malha 2022) para o CRS métrico da cidade.
2. Faz o *overlay* de interseção entre as duas malhas.
3. Remove fatias residuais (*slivers*) abaixo de `LIMIAR_SLIVER_M2` — ruído de topologia do overlay, não
   sobreposição geográfica real.
4. Calcula o peso de cada fatia: `área_fatia / Σ(áreas das fatias do mesmo setor de origem)`. Essa normalização
   (em vez de dividir pela área total do setor de origem) é o que **preserva o total municipal**: a área do
   setor de origem que cai fora da malha 2022 é redistribuída proporcionalmente entre as fatias que *estão*
   dentro, em vez de ser perdida.
5. Variáveis **extensivas** (contagens): soma ponderada pelas fatias, agregada por setor de destino (2022).
6. **V24** (renda, intensiva): média ponderada por `V26 × peso`, conforme decisão do projeto.

In [ ]:
def compatibiliza_ano(cidade, ano_origem):
    slug = CIDADES[cidade]["slug"]
    crs_m = CRS_METRICO[cidade]

    origem = gpd.read_file(IN_DIR / f"{slug}_{ano_origem}.gpkg").to_crs(crs_m)
    destino = gpd.read_file(IN_DIR / f"{slug}_2022.gpkg").to_crs(crs_m)
    destino = destino.rename(columns={"id_setor_origem": "CD_SETOR"})

    colunas_origem = ["id_setor_origem", "geometry"] + VARS_EXTENSIVAS + [VAR_RENDA]
    origem_min = origem[colunas_origem].copy()

    inter = gpd.overlay(origem_min, destino[["CD_SETOR", "geometry"]], how="intersection", keep_geom_type=True)
    inter["area_inter_m2"] = inter.geometry.area
    n_antes = len(inter)
    inter = inter[inter["area_inter_m2"] > LIMIAR_SLIVER_M2].copy()
    n_slivers_removidos = n_antes - len(inter)

    soma_area_origem = inter.groupby("id_setor_origem")["area_inter_m2"].transform("sum")
    inter["peso"] = inter["area_inter_m2"] / soma_area_origem

    for col in VARS_EXTENSIVAS:
        inter[f"{col}__pond"] = inter[col].fillna(0) * inter["peso"]

    agregado = inter.groupby("CD_SETOR")[[f"{c}__pond" for c in VARS_EXTENSIVAS]].sum()
    agregado.columns = VARS_EXTENSIVAS

    inter["peso_renda"] = inter[PESO_RENDA].fillna(0) * inter["peso"]
    inter["renda_x_peso"] = inter[VAR_RENDA].fillna(0) * inter["peso_renda"]
    soma_peso_renda = inter.groupby("CD_SETOR")["peso_renda"].sum()
    soma_renda_pond = inter.groupby("CD_SETOR")["renda_x_peso"].sum()
    # float("nan") em vez de pd.NA: pd.NA sobrevive a um round-trip por GeoPackage como a *string* "<NA>",
    # não como nulo — o que quebraria qualquer conta feita depois de reler o arquivo (ver notebook 02, achado
    # de harmonização de tipos).
    agregado[VAR_RENDA] = (soma_renda_pond / soma_peso_renda.replace(0, float("nan"))).reindex(agregado.index)

    # Reindexa para incluir todos os setores de destino, mesmo sem nenhuma interseção com a origem.
    # Extensivas ausentes -> 0 (sem cobertura histórica é um zero plausível: área que não existia/estava vazia).
    # V24 (intensiva) ausente -> NaN (renda desconhecida não é o mesmo que renda zero).
    agregado = agregado.reindex(destino["CD_SETOR"])
    agregado[VARS_EXTENSIVAS] = agregado[VARS_EXTENSIVAS].fillna(0)
    agregado.index.name = "CD_SETOR"

    cobertura = inter.groupby("CD_SETOR")["area_inter_m2"].sum() / destino.set_index("CD_SETOR").geometry.area
    diagnostico = {
        "cidade": cidade, "ano_origem": ano_origem,
        "n_fatias_overlay": n_antes, "n_slivers_removidos": n_slivers_removidos,
        "n_setores_destino": len(destino), "cobertura_media": float(cobertura.mean()),
        "cobertura_minima": float(cobertura.min()),
    }
    return agregado.reset_index(), diagnostico


## 3. Execução para as 5 cidades × 2 anos de origem (2000, 2010)

In [ ]:
resultados = {}
diagnosticos = []

for cidade in CIDADES:
    for ano_origem in (2000, 2010):
        agregado, diag = compatibiliza_ano(cidade, ano_origem)
        resultados[(cidade, ano_origem)] = agregado
        diagnosticos.append(diag)
        print(f"{cidade} {ano_origem}: {diag['n_setores_destino']} setores destino, "
              f"cobertura média {diag['cobertura_media']:.3f}, mínima {diag['cobertura_minima']:.3f}")

df_diagnostico = pd.DataFrame(diagnosticos)
df_diagnostico


Cobertura média próxima de 1 indica que praticamente toda a área de cada setor de 2022 recebeu contribuição de
algum setor de origem. Cobertura mínima baixa em setores isolados costuma refletir expansão urbana posterior ao
ano de origem (setor 2022 que não existia, ou existia como área vazia, em 2000/2010) — nesse caso o valor
resultante é 0, o que é o comportamento correto.

## 4. Validação: totais municipais preservados

- Compara a soma municipal de cada variável na malha de 2022 (após interpolação) com o total observado
  diretamente na malha de origem (baseline do notebook 01).
- Premissa: a normalização do peso (passo 4 da função) deveria manter essa diferença próxima de zero — o que
  sobra é limitado a slivers removidos e à perda de área de setores de origem inteiramente fora do território
  de 2022 (ex.: Santarém — ver notebook 02).

In [ ]:
linhas_validacao = []
for cidade in CIDADES:
    for ano_origem in (2000, 2010):
        agregado = resultados[(cidade, ano_origem)]
        base = baseline[(baseline["cidade"] == cidade) & (baseline["ano"] == ano_origem)].iloc[0]
        for var in ["V1", "V26", "V27"]:
            total_interpolado = agregado[var].sum()
            total_original = base[var]
            dif_pct = 100 * (total_interpolado - total_original) / total_original if total_original else float("nan")
            linhas_validacao.append({
                "cidade": cidade, "ano_origem": ano_origem, "variavel": var,
                "total_original": round(total_original, 1),
                "total_interpolado_2022": round(total_interpolado, 1),
                "diferenca_%": round(dif_pct, 2),
            })

df_validacao = pd.DataFrame(linhas_validacao)
df_validacao


- Diferenças acima de poucos pontos percentuais devem ser cruzadas com o notebook 02 (qualidade geométrica)
  antes de prosseguir — não são erro de arredondamento.
- Faixa esperada: 2010→2022 ~0% (malhas quase inalteradas); 2000→2022 tipicamente entre -0,2% e -1,9%
  (generalização de limite e área residual de rural que não fecha perfeitamente com 2022).
- **Santarém é a exceção conhecida** (-6,6% em 2000, -4,1% em 2010): a redução territorial real de 2012 (ver
  notebook 02) tira setores de origem inteiramente fora da malha de 2022 — sem interseção, não entram no
  `groupby` do passo 4 (seção 2), e sua população simplesmente não aparece na malha de 2022, em vez de ser
  redistribuída para os setores remanescentes.

## 5. Montagem da camada larga por cidade

Uma única tabela por cidade, geometria = malha de 2022, com as três séries de colunas (`_2000`, `_2010`, `_2022`).
Para 2022 não há interpolação: os valores são os do próprio ano, lidos diretamente do arquivo padronizado.

In [ ]:
for cidade in CIDADES:
    slug = CIDADES[cidade]["slug"]
    destino = gpd.read_file(IN_DIR / f"{slug}_2022.gpkg")
    destino = destino.rename(columns={"id_setor_origem": "CD_SETOR"})

    tabela = destino[["CD_SETOR", "geometry", "area_m2"] + VARS_EXTENSIVAS + [VAR_RENDA, "V24A"]].copy()
    tabela = tabela.rename(columns={c: f"{c}_2022" for c in VARS_EXTENSIVAS + [VAR_RENDA, "V24A"]})

    for ano_origem in (2000, 2010):
        agregado = resultados[(cidade, ano_origem)].rename(
            columns={c: f"{c}_{ano_origem}" for c in VARS_EXTENSIVAS + [VAR_RENDA]}
        )
        tabela = tabela.merge(agregado, on="CD_SETOR", how="left")

    tabela = gpd.GeoDataFrame(tabela, geometry="geometry", crs=destino.crs)
    tabela.to_file(OUT_DIR / f"{slug}_contagens.gpkg", driver="GPKG")
    print(f"{cidade}: {len(tabela)} setores, {tabela.shape[1]} colunas -> {slug}_contagens.gpkg")
